In [6]:
# STEP 1: Import Libraries

import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, GridSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score


# STEP 2: Load Breast Cancer Dataset

dataset = load_breast_cancer()

X = dataset.data
y = dataset.target

print("Dataset loaded successfully")
print("Total Records:", X.shape[0])
print("Total Features:", X.shape[1])

print("\nTarget Classes:")
print(dataset.target_names)


# STEP 3: Create Train, Validation and Test Splits

# First split: 80% training + validation, 20% testing

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Second split: 60% training, 20% validation

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

print("\nData Split:")
print("Training Records:", len(X_train))
print("Validation Records:", len(X_val))
print("Testing Records:", len(X_test))


# STEP 4: Train Logistic Regression Baseline Model

model = LogisticRegression(max_iter=5000)

model.fit(X_train, y_train)

y_pred = model.predict(X_val)

baseline_accuracy = accuracy_score(y_val, y_pred)

print("\nBaseline Logistic Regression")
print("Validation Accuracy:", baseline_accuracy)


# STEP 5: Perform 5-Fold Cross Validation

cv_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

print("\n5-Fold Cross Validation Scores:")
print(cv_scores)

cv_mean = cv_scores.mean()

print("Cross Validation Mean:", cv_mean)


# STEP 6: Create Pipeline with StandardScaler and Logistic Regression

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=5000))
])

pipeline.fit(X_train, y_train)

pipeline_pred = pipeline.predict(X_val)

pipeline_accuracy = accuracy_score(y_val, pipeline_pred)

print("\nPipeline Model")
print("Pipeline Validation Accuracy:", pipeline_accuracy)


# STEP 7: Hyperparameter Tuning using GridSearchCV

param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)

print("\nBest Parameters:")
print(grid_search.best_params_)

print("Best Cross Validation Score:")
print(grid_search.best_score_)


# STEP 8: Evaluate Tuned Model on Validation Data

best_model = grid_search.best_estimator_

tuned_pred = best_model.predict(X_val)

tuned_accuracy = accuracy_score(y_val, tuned_pred)

print("\nTuned Model Results")
print("Tuned Validation Accuracy:", tuned_accuracy)


# STEP 9: Final Test Evaluation

test_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, test_pred)

print("\nFinal Test Accuracy:", test_accuracy)


# STEP 10: Comparison Table

results = pd.DataFrame({
    'Model': [
        'Baseline Logistic Regression',
        '5-Fold Cross Validation Mean',
        'StandardScaler + Logistic Regression',
        'Tuned Logistic Regression'
    ],
    'Accuracy': [
        baseline_accuracy,
        cv_mean,
        pipeline_accuracy,
        tuned_accuracy
    ]
})

print("\nMODEL COMPARISON TABLE")
print(results.to_string(index=False))

print("\nPractical Completed Successfully!")

Dataset loaded successfully
Total Records: 569
Total Features: 30

Target Classes:
['malignant' 'benign']

Data Split:
Training Records: 341
Validation Records: 114
Testing Records: 114

Baseline Logistic Regression
Validation Accuracy: 0.9473684210526315

5-Fold Cross Validation Scores:
[0.97101449 0.95588235 0.98529412 0.94117647 0.95588235]
Cross Validation Mean: 0.9618499573742542

Pipeline Model
Pipeline Validation Accuracy: 0.9912280701754386

Best Parameters:
{'classifier__C': 1}
Best Cross Validation Score:
0.9765132139812447

Tuned Model Results
Tuned Validation Accuracy: 0.9912280701754386

Final Test Accuracy: 0.9824561403508771

MODEL COMPARISON TABLE
                               Model  Accuracy
        Baseline Logistic Regression  0.947368
        5-Fold Cross Validation Mean  0.961850
StandardScaler + Logistic Regression  0.991228
           Tuned Logistic Regression  0.991228

Practical Completed Successfully!
